# RV Carry Pipeline — 4 signals on swaption calendars

Tests the full pipeline end to end on **synthetic** data (no real market data here on purpose — swap in real historical panels later, same shape: dates x (tenor, expiry)).

Pipeline, per calendar (e.g. `2Y30Y - 1Y30Y`):

1. **Signal 1 — raw carry z-score**: price the calendar on spot prices, z-score the level against its own history.
2. **Signal 2 — DF-normalized z-score**: divide the calendar by the front-leg price ("discount" it by the other leg), z-score that.
3. **Signal 3 — PCA residual z-score (raw)**: PCA-denoise the whole calendar surface for the tenor (all consecutive-expiry calendars, across history), residual = actual − PCA reconstruction, z-score the residual.
4. **Signal 4 — PCA residual z-score (DF)**: same as 3, run on the DF-normalized surface instead of the raw one.

`composite_score` is a simple average of the 4 z-scores; `signals_agree` flags when all 4 point the same way.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rv_carry.synthetic_data import generate_synthetic_surface
from rv_carry.calendars import price_surface, calendar_grid, normalized_calendar_grid
from rv_carry.pipeline import run_rv_carry_pipeline

## 1. Data

`generate_synthetic_surface` builds 252 business days of forward/vol/BPV panels across tenors x expiries, with a common level+slope factor (so PCA has real structure to find) plus idiosyncratic noise per cell.

It also injects a synthetic anomaly: **richens the 1Y expiry vol at the 30Y tenor over the last 10 days** — a stand-in for the "1Y30Y looks rich" scenario from the strategy discussion, so the demo signals have something concrete to flag instead of pure noise.

**When you're back on your other laptop**: replace this cell with your real historical panels. They just need to be `DataFrame`s indexed by date, with `MultiIndex` columns `(tenor, expiry)` — same shape as what this function returns. Nothing downstream needs to change.

In [ ]:
forward_panel, vol_panel, bpv_panel, expiry_years = generate_synthetic_surface(
    n_days=252, seed=1, inject_anomaly=True
)

print("panel shape (dates x tenor*expiry cells):", forward_panel.shape)
vol_panel.tail(3)

## 2. Price surface + calendar surfaces

`price_surface` prices every (tenor, expiry) cell ATM via Bachelier. `calendar_grid` builds all consecutive-expiry calendars for one tenor (e.g. `1Y-6M`, `2Y-1Y`, `3Y-2Y`, `5Y-3Y` on the 30Y tenor) — this is the "calendar surface" signals 3/4 denoise with PCA. `normalized_calendar_grid` is the same thing divided by each calendar's front-leg price (the DF normalization from signal 2).

In [ ]:
prices = price_surface(forward_panel, vol_panel, bpv_panel, expiry_years)

TENOR = 30
EXPIRIES_ORDERED = ["6M", "1Y", "2Y", "3Y", "5Y"]

cal_grid = calendar_grid(prices, TENOR, EXPIRIES_ORDERED)
norm_grid = normalized_calendar_grid(prices, TENOR, EXPIRIES_ORDERED)

cal_grid.tail(5)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
cal_grid.plot(ax=ax)
ax.set_title(f"Raw calendar surface, {TENOR}Y tenor")
ax.set_ylabel("calendar price")
ax.axvline(cal_grid.index[-10], color="grey", linestyle="--", linewidth=1, label="anomaly window starts")
ax.legend()
plt.show()

## 3. Run the 4-signal pipeline on one calendar

Worked example: `2Y30Y - 1Y30Y`.

In [ ]:
result = run_rv_carry_pipeline(
    price_panel=prices,
    tenor=TENOR,
    expiry_front="1Y",
    expiry_back="2Y",
    calendar_panel_for_tenor=cal_grid,
    normalized_panel_for_tenor=norm_grid,
    n_components=2,
)

for k, v in result.items():
    if not k.startswith("_"):
        print(f"{k:32s}: {v}")

Signal 3's residual series (actual raw calendar minus what the top-2 PCA factors alone would predict) — this is what gets z-scored:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
result["_residual_3"].plot(ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Signal 3 residual: 2Y30Y-1Y30Y actual vs. PCA-implied")
plt.show()

## 4. Scan every calendar on the tenor — check for dispersion

This is the sanity check that matters most: with a naive single-signal approach every cell tended to point the same way (the beta problem from the strategy discussion). With the 4-signal composite, cells should genuinely disagree with each other — and the cell containing the injected anomaly (`2Y30-1Y30`, since the front leg 1Y30Y was richened) should stand out.

In [ ]:
rows = []
for front, back in zip(EXPIRIES_ORDERED[:-1], EXPIRIES_ORDERED[1:]):
    r = run_rv_carry_pipeline(prices, TENOR, front, back, cal_grid, norm_grid, n_components=2)
    rows.append({k: v for k, v in r.items() if not k.startswith("_")})

summary = pd.DataFrame(rows).set_index("calendar")
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
summary["composite_score"].plot(kind="bar", ax=ax,
                                  color=["#e34948" if v > 0 else "#2a78d6" for v in summary["composite_score"]])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("composite z-score")
ax.set_title(f"Composite carry signal across the {TENOR}Y tenor column\n(red = rich/sell, blue = cheap/buy)")
plt.tight_layout()
plt.show()

## 5. Next steps

- Swap `generate_synthetic_surface(...)` for real historical forward/vol/BPV panels (same `DataFrame` shape) once you're back on your other laptop.
- Loop this scan across every tenor column (2Y, 5Y, 10Y, 30Y, ...) to get the full macro grid, not just one tenor.
- Once a calendar is selected (large composite score, `signals_agree=True`), hand the two legs to `breakeven_solver.py` (already in this repo) to fine-tune the strike of each leg — that tool stays exactly as-is, it just now runs on the output of this selection step instead of on a raw, unfiltered grid.